# Regularisation and tuning neural networks — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D019_01_Model_Optimization_Tuning.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# <center>C8: Model Optimization and Tuning</center>

## 1. Optimization Challenges

Training Deep Networks is hard. Aside from selecting the right architecture, we often face "pathologies" that stop learning.

### 1.1 The Vanishing Gradient Problem
In deep networks, gradients are multiplied via the Chain Rule during Backpropagation. If these gradients are small (< 1), they shrink exponentially as they go back to the first layers. The result? **Early layers stop learning**.

![Vanishing Gradient Diagram](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Faiml.com%2Fwp-content%2Fuploads%2F2023%2F11%2Fvanishing-and-exploding-gradient-1.png&f=1&nofb=1&ipt=04f8bc8be69fd32fa79ce4f35a8a2005f67168ce5a2cc616ada9229a432d392f)
![Vanishing Gradient Neural Diagram](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmiro.medium.com%2Fv2%2Fresize%3Afit%3A716%2F1*-rPcuafUSo-SwZF121y_rQ.png&f=1&nofb=1&ipt=6df49acd41f35a191a1e7d93af1a71175672a8ef3348d6cd2d5201a081c09cb8)

*As gradients flow backward (right to left), they become thinner (smaller), leaving initial layers unchanged.*

### 1.2 The Dying ReLU Problem
ReLU ($max(0, x)$) is great, but if a neuron's weights are updated such that it always outputs 0 (negative input), it will **never activate again**. Its gradient is 0 forever. It is "dead".

![Dying ReLU](https://aiml.com/wp-content/uploads/2023/09/ReLU-Leaky-ReLU-ELU-1024x794.png)

**Solutions**:
-   **Leaky ReLU**: Allows a small gradient when $x < 0$ ($y = 0.01x$).
-   **He Initialization**: Smart weight initialization.

## 2. Model Validation Strategy

We use a **Validation Set** to proxy performance on unseen data. The mostly critical concept here is **Overfitting**.

### Early Stopping
We stop training when the Validation Loss stops decreasing, even if Training Loss is still going down.

![Early Stopping Curve](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2Fdfa532f08c7b0720c72306d327a82f78f7e17f5d-1000x843.png&w=2048&q=75)

*The "Sweet Spot" is where the gap between Training and Validation loss starts to widen.*

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop if val_loss doesn't improve for 3 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, verbose=1)

# model.fit(..., callbacks=[early_stop])

In [ ]:
# ═══ CHECKPOINT — "Dropout Regularization" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, verbose=1)

print("Checkpoint restored. Ready for: Dropout Regularization")

## 3. Dropout Regularization

**Overfitting** is when a model performs great on training data but poorly on unseen data. It "memorizes" rather than "learns".

**Dropout** is a powerful regularization technique that randomly "drops" (sets to zero) a percentage of neurons during training. This forces the network to learn robust features and prevents it from relying too heavily on any single neuron.

![Dropout Visualization](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.baeldung.com%2Fwp-content%2Fuploads%2Fsites%2F4%2F2020%2F05%2F2-1-2048x745-1.jpg&f=1&nofb=1&ipt=a11c3f79fa65d08341ebc34f2b667f43ffe55155331486d01a2bf9bf34103efb)

### How It Works
- During **training**: Randomly drop neurons with probability `p` (e.g., 0.3 means 30% dropout)
- During **inference**: Use all neurons (no dropout)

### Best Practices
- Start with 0.2-0.5 dropout rate
- Place after Dense layers, before activation (or after)
- Higher dropout for larger networks
- Don't use dropout in the output layer

In [ ]:
# TODO: compute `n_redundant`, `model_no_dropout`, `loss` from `accuracy`, `val_accuracy`, `loss`
# Name your result: n_redundant, model_no_dropout, loss, metrics

### Analysis

**What to observe:**
1. **Without Dropout**: Training accuracy quickly reaches ~100%, but validation accuracy plateaus lower → **Overfitting!**
2. **With Dropout**: Training accuracy grows slower, but validation accuracy is closer → **Better generalization!**

**The Gap**: The difference between training and validation accuracy is the "overfitting gap". Dropout reduces this gap.

**Trade-off**: Dropout slightly reduces training accuracy but significantly improves test performance. This is exactly what we want!

## 4. Hyperparameter Tuning

Modern Deep Learning has too many knobs to turn manually (Learning Rate, Units, Layers, Dropout, etc.).

### Search Strategies
-   **Grid Search**: Tries EVERY combination. Slow but exhaustive.
-   **Random Search**: Tries random combinations. Surprisingly effective and much faster.

![Grid vs Random Search](https://substackcdn.com/image/fetch/$s_!29ot!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F1bf0f3af-22ce-46b1-a0a5-d34ee4544514_772x441.png)

*Random search (right) often finds good parameters faster because it explores more unique values for each important parameter.*

## 5. Tuning with Keras Tuner

Instead of writing manual `for` loops, we use **Keras Tuner**.

*(Note: You may need to run `!pip install keras-tuner`)*

In [ ]:
# !pip install -q keras-tuner

### 4.1 Basic Keras Tuner Example (CORRECTED)

In [ ]:
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

# 1. Define a "Hypermodel" function
# This function now accepts input_dim as a parameter to make it flexible
def build_model(hp, input_dim):
    model = keras.Sequential()
    
    # Tune the number of units in the first Dense layer
    # Choose an optimal value between 32-512
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    
    # Use the input_dim parameter instead of hardcoded value
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(input_dim,)))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001, or 0.0001
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model

# Example usage with actual data:
# Assuming X_train has shape (n_samples, n_features)
# input_dim = X_train.shape[1]

# 2. Initialize the Tuner (Hyperband is a popular algorithm)
# Note: You need to pass input_dim when initializing
# tuner = kt.Hyperband(
#     lambda hp: build_model(hp, input_dim=X_train.shape[1]),
#     objective='val_accuracy',
#     max_epochs=10,
#     factor=3,
#     directory='my_dir', overwrite=True,
#     project_name='intro_to_kt'
# )

# 3. Run the search with early stopping to avoid wasting time
# stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
# tuner.search(X_train, y_train, epochs=50, validation_data=(X_val, y_val), callbacks=[stop_early])

# 4. Get the best hyperparameters
# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
# print(f"Best units: {best_hps.get('units')}")
# print(f"Best learning rate: {best_hps.get('learning_rate')}")

# 5. Build and train the best model
# best_model = tuner.hypermodel.build(best_hps)
# history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val))

### 4.2 Complete Example with Sample Data

In [ ]:
# TODO: compute `X`, `y`, `model` from `accuracy`
# Name your result: X, y, model, hp_units

In [ ]:
# ═══ CHECKPOINT — "Advanced Tuning: Multiple Layers and Architecture Search" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, verbose=1)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
X, y = make_classification(n_samples=500, n_features=50, n_informative=30, n_redundant=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model_no_dropout = keras.Sequential([layers.Dense(128, activation='relu', input_shape=(50,)), layers.Dense(64, activation='relu'), layers.Dense(32, activation='relu'), layers.Dense(1, activation='sigmoid')])
model_no_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_with_dropout = keras.Sequential([layers.Dense(128, activation='relu', input_shape=(50,)), layers.Dropout(0.3), layers.Dense(64, activation='relu'), layers.Dropout(0.3), layers.Dense(32, activation='relu'), layers.Dropout(0.2), layers.Dense(1, activation='sigmoid')])
model_with_dropout.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_no_dropout = model_no_dropout.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=1, batch_size=32, verbose=0)
history_with_dropout = model_with_dropout.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=1, batch_size=32, verbose=0)
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers
def build_model(hp, input_dim):
    model = keras.Sequential()
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(input_dim,)))
    model.add(layers.Dense(1, activation='sigmoid'))
    hp_learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model
import numpy as np
from sklearn.model_selection import train_test_split
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers
X = np.random.randn(1000, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
def build_tunable_model(hp):
    model = keras.Sequential()
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(X_train.shape[1],)))
    hp_dropout = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
    model.add(layers.Dropout(hp_dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    hp_learning_rate = hp.Choice('learning_rate', values=[0.01, 0.001, 0.0001])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate), loss='binary_crossentropy', metrics=['accuracy'])
    return model
tuner = kt.Hyperband(build_tunable_model, objective='val_accuracy', max_epochs=1, factor=3, directory='my_dir', overwrite=True, project_name='complete_example')
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
tuner.search(X_train, y_train, epochs=1, validation_data=(X_val, y_val), callbacks=[stop_early], verbose=1)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train, y_train, epochs=1, validation_data=(X_val, y_val), verbose=0)
val_accuracy = best_model.evaluate(X_val, y_val, verbose=0)[1]

print("Checkpoint restored. Ready for: Advanced Tuning: Multiple Layers and Architecture Search")

### 4.3 Advanced Tuning: Multiple Layers and Architecture Search

In [ ]:
# TODO: compute `model`, `num_layers`, `hp_units` from `accuracy`
# Name your result: model, num_layers, hp_units, hp_learning_rate

## 5. Exercises

**Exercise 1: Addressing the Dying ReLU**
Create a simple model but use `LeakyReLU` instead of standard `ReLU`. You can import it via `from tensorflow.keras.layers import LeakyReLU`.

**Exercise 2: Tuning the Titanic**
Load your Titanic dataset. Use **Keras Tuner** to find the best:
1. Number of layers (1, 2, or 3).
2. Dropout rate (0.2, 0.3, 0.5).
What is the best architecture found?

### Exercise 1 Solution: LeakyReLU Model

### Exercise 2 Solution: Tuning Titanic Dataset